# 10. Subscription은 문맥, Mandate는 권한, Assignment는 실행 시도

원본: https://github.com/deepseek-ai/dsh-superagent-prototype/tree/8c32a9dc5d72c033816c6a8110b37f40cf94fba4

근거 파일: `src/core/intake.ts`, `tickets.ts`, `assignments.ts`; `tests/core-engine.spec.ts`, `followups.spec.ts`

**범위:** 원본은 TypeScript/Cordis입니다. 아래는 원본의 일부 불변조건을 Python/LangGraph로 실험하며 공식 API 호환 구현이 아닙니다. 새 커널에서 순서대로 실행합니다. 외부 API 호출이 없습니다.

## 하나의 상태 변수로 표현할 수 없는 이유
Ticket은 승인된 일이며 Assignment는 그 일을 수행하는 실행 시도입니다. Ticket 하나에 Assignment가 여러 번 생길 수 있습니다. `blocked/failed` 보고는 작업 전체의 권한 철회와 다릅니다. 원본에서 사람 stop은 Mandate를 철회하므로 다시 전달하거나 자동 resume하지 않습니다.

아래는 사람 후속 메시지, bot, context-only 구독을 비교하는 축소 모델입니다. SourceScope 수신 필터·트랜잭션·원본의 모든 종료 규칙은 생략했습니다.

In [1]:
from dataclasses import dataclass, field
@dataclass
class Ticket:
    status: str='open'
    ended_by: str|None=None
    mandate_valid: bool=True
    assignment: int=1
    inputs: list[str]=field(default_factory=list)
    reports: list[str]=field(default_factory=list)

def receive(t, text, *, human=True, context_only=False):
    if t.ended_by=='human': return {'delivered':False,'eval':True}
    if t.status=='ended':
        if not human or context_only or not t.mandate_valid:
            return {'delivered':False,'eval':True}
        t.status='open'; t.ended_by=None; t.assignment+=1
    t.inputs.append(text)
    return {'delivered':True,'eval':not human or context_only}
def report(t, assignment, text):
    if assignment!=t.assignment or t.status!='open': return False
    t.reports.append(text); return True

t=Ticket()
assert receive(t,'범위 내 후속 요청')=={'delivered':True,'eval':False}
assert receive(t,'CI 실패',human=False)=={'delivered':True,'eval':True}
assert receive(t,'관련 대화',context_only=True)=={'delivered':True,'eval':True}
t.status='ended'; t.ended_by='executor'
assert not receive(t,'CI 재시도',human=False)['delivered']
assert t.status=='ended'
assert receive(t,'다시 확인해 주세요')['delivered'] and t.assignment==2
assert not report(t,1,'이전 실행이 늦게 반환한 완료')
assert report(t,2,'현재 실행 보고')
print('작업은 유지, 실행 시도는 교체:',t)

작업은 유지, 실행 시도는 교체: Ticket(status='open', ended_by=None, mandate_valid=True, assignment=2, inputs=['범위 내 후속 요청', 'CI 실패', '관련 대화', '다시 확인해 주세요'], reports=['현재 실행 보고'])


In [2]:
stopped=Ticket(status='ended',ended_by='human',mandate_valid=False)
assert not receive(stopped,'새 댓글')['delivered']
assert stopped.status=='ended'
context=Ticket(status='ended',ended_by='executor')
assert not receive(context,'참고용 메시지',context_only=True)['delivered']
revoked=Ticket(status='ended',ended_by='executor',mandate_valid=False)
assert not receive(revoked,'resume 요청')['delivered']
print('사람 stop / context-only / 재검증 실패는 자동 resume하지 않음')

사람 stop / context-only / 재검증 실패는 자동 resume하지 않음


## 원본의 더 정교한 조건
원본은 delivered와 model history에 들어간 read도 구분합니다. 재시작 후 unread 입력을 재전달하되 이미 처리된 외부 작업은 점검하도록 recovery note를 보냅니다. 정책 변경은 진행 중 Ticket을 일괄 취소하는 대신 변경 내용을 전달하며, ended Ticket의 resume 시 현재 정책으로 다시 검증합니다. PR의 ready 보고와 PR merge/close도 같은 종료 조건이 아닙니다.

## 과제와 해설
과제: Thread 둘이 하나의 Ticket에 연결된 경우 동일 Trigger가 두 번 전달되지 않게 만드세요. 해설: 원본 Intake의 reached Map은 Ticket ID별로 합치며, 한 경로라도 소유 구독이면 context-only로 잘못 분류하지 않습니다. Correlation만 있는 Thread로는 라우팅하지 않아야 합니다.